# Off-Label Pixeltable: Help My Parents Pick a Front Door Color

Pixeltable is a multimodal backend for AI apps. This video uses it off label, for a family project.

My parents want to paint their front door navy. Benjamin Moore has dozens of navies, and the little chips at the store all look the same. So I'm going to put the whole decision in one Pixeltable table: one photo of a front door, and one row per paint color.

I type in six color numbers. Pixeltable looks up each color's official name, hex code, and LRV on Benjamin Moore's site, draws a paint chip, and has Gemini repaint just the door in that color. At the end we get one table, one row per color, that my parents can actually look at and argue about.

## Start from an empty `door_demo` directory

Start Jupyter with `uv run jupyter lab` in this folder. Gemini needs `GEMINI_API_KEY` in your environment.

In [ ]:
!uv sync

In [ ]:
import pixeltable as pxt

# Start fresh: drop the tables from any earlier run.
pxt.drop_dir('door_demo', force=True, if_not_exists='ignore')

## Two tables: one door, many paints

`doors` holds the photo of the front door. In `paints`, the only column you fill in is the color number, such as `HC-154`. Everything else is a computed column:

1. `info` is the color's official data from its Benjamin Moore page. `name`, `hex`, and `lrv` come out of it as their own columns.
2. `bm_photo` is Benjamin Moore's photo of the color, and `chip` is a paint chip drawn from the official hex.
3. `painted_door` is the door photo, repainted by Gemini in that color. Gemini gets the door photo, the chip, and a prompt that names the color.

In [ ]:
!sed -n '/^class Doors/,$p' schema.py

`pxt schema diff` shows the plan, and `pxt schema update` creates the tables.

In [ ]:
!pxt schema diff schema.py door_demo

In [ ]:
!pxt schema update schema.py door_demo -f

In [ ]:
doors = pxt.get_table('door_demo.doors')
paints = pxt.get_table('door_demo.paints')

## Insert the front door

The photo is a free photo by Shaylyn on [Unsplash](https://unsplash.com/photos/-zNNFDR3dPM). Pixeltable downloads it from the URL when it inserts the row. To try your own door, put your photo's path or URL here.

In [ ]:
doors.insert([{
    'door': 'front',
    'photo': 'https://images.unsplash.com/photo-1601041597271-71988152f98b?w=1600&q=85&fm=jpg',
    'credit': 'Photo by Shaylyn on Unsplash: https://unsplash.com/photos/-zNNFDR3dPM',
}])
doors.select(doors.door, doors.photo, doors.credit).collect()

## Insert six navy paint colors

All I insert is six color numbers. `on_error='ignore'` keeps the insert going if a model call fails on a row, and saves the error on that row instead.

In [ ]:
navies = ['HC-154', 'HC-156', 'HC-155', '2062-20', '1680', '2060-10']
paints.insert([{'number': n} for n in navies], on_error='ignore')

## Compare the official color data

LRV is light reflectance value: how much light a color reflects, from 0 (black) to 100 (white). On a door, a lower LRV reads darker and closer to black.

In [ ]:
paints.select(
    paints.name, paints.number, paints.hex, paints.lrv, paints.collection, paints.best_selling, paints.chip
).order_by(paints.lrv, asc=False).collect()

## Rerun only the doors that failed

Sometimes Gemini returns no image. `painted_door.errortype` and `painted_door.errormsg` show which colors failed and why. `IMAGE_RECITATION` means Gemini judged its output too close to an image it knows. It varies from call to call, so trying again usually works.

In [ ]:
failed = paints.painted_door.errortype != None
paints.where(failed).select(paints.name, paints.number, paints.painted_door.errormsg).collect()

The error shows up in `painted_door`, but the refusal is stored in `response`, the Gemini call that `painted_door` reads. So I recompute `response` on just the failed rows. Pixeltable then recomputes `painted_door` for those rows too. The other doors stay as they are, and you don't pay for them again.

In [ ]:
if paints.where(failed).count() > 0:
    paints.recompute_columns('response', where=failed)
paints.where(failed).count()

## One door, six navies

In [ ]:
paints.select(
    paints.name, paints.number, paints.hex, paints.lrv, paints.chip, paints.painted_door
).order_by(paints.lrv, asc=False).collect()

## Add one more color

New rows are the only ones that run. Here I add Benjamin Moore's Deep Royal, and Pixeltable reads its data, draws its chip, and paints the door for that one row. Then it joins the comparison, sorted by LRV.

In [ ]:
paints.insert([{'number': '2061-10'}], on_error='ignore')
paints.select(
    paints.name, paints.number, paints.hex, paints.lrv, paints.chip, paints.painted_door
).order_by(paints.lrv, asc=False).collect()

## Other off-label ideas

Nothing here is about doors. The pattern is: one photo you care about, a list of options, official facts for each option, and an AI preview of each option on your photo. A few more you could build from this notebook by changing the photo, the color numbers, and one sentence of the prompt:

- Paint colors for a bedroom wall, or for the shutters and trim
- Kitchen backsplash tiles, using photos of the tile samples instead of paint chips
- Hair colors for a selfie, before a salon appointment

Everything stays in Pixeltable under `~/.pixeltable`. When my parents call back a week later and ask about the second-lightest blue again, the table is still there: `pxt.get_table('door_demo.paints')`, or browse it with `pxt dashboard`.